---

_You are currently looking at **version 1.1** of this notebook. To download notebooks and datafiles, as well as get help on Jupyter notebooks in the Coursera platform, visit the [Jupyter Notebook FAQ](https://www.coursera.org/learn/python-social-network-analysis/resources/yPcBs) course resource._

---

# Assignment 1 - Creating and Manipulating Graphs

Eight employees at a small company were asked to choose 3 movies that they would most enjoy watching for the upcoming company movie night. These choices are stored in the file `Employee_Movie_Choices.txt`.

A second file, `Employee_Relationships.txt`, has data on the relationships between different coworkers. 

The relationship score has value of `-100` (Enemies) to `+100` (Best Friends). A value of zero means the two employees haven't interacted or are indifferent.

Both files are tab delimited.

In [3]:
import networkx as nx
import pandas as pd
import numpy as np
from networkx.algorithms import bipartite


# This is the set of employees
employees = set(['Pablo',
                 'Lee',
                 'Georgia',
                 'Vincent',
                 'Andy',
                 'Frida',
                 'Joan',
                 'Claude'])

# This is the set of movies
movies = set(['The Shawshank Redemption',
              'Forrest Gump',
              'The Matrix',
              'Anaconda',
              'The Social Network',
              'The Godfather',
              'Monty Python and the Holy Grail',
              'Snakes on a Plane',
              'Kung Fu Panda',
              'The Dark Knight',
              'Mean Girls'])


# you can use the following function to plot graphs
# make sure to comment it out before submitting to the autograder
def plot_graph(G, weight_name=None):
    '''
    G: a networkx G
    weight_name: name of the attribute for plotting edge weights (if G is weighted)
    '''
    %matplotlib notebook
    import matplotlib.pyplot as plt
    
    plt.figure()
    pos = nx.spring_layout(G)
    edges = G.edges()
    weights = None
    
    if weight_name:
        weights = [int(G[u][v][weight_name]) for u,v in edges]
        labels = nx.get_edge_attributes(G,weight_name)
        nx.draw_networkx_edge_labels(G,pos,edge_labels=labels)
        nx.draw_networkx(G, pos, edges=edges, width=weights);
    else:
        nx.draw_networkx(G, pos, edges=edges);

### Question 1

Using NetworkX, load in the bipartite graph from `Employee_Movie_Choices.txt` and return that graph.

*This function should return a networkx graph with 19 nodes and 24 edges*

In [14]:
def answer_one():
        
    # Your Code Here
    input = open("Employee_Movie_Choices.txt", "r")
    lines =  input.read().splitlines()
    G = nx.Graph()
    for line in lines[1:]:
        A, B = line.split('\t')
        #print(A,'|', B)
        G.add_edge(A,B)
    #print (len(G.nodes()), len(G.edges()))    
    return G # Your Answer Here
#G = answer_one()
#plot_graph(G)

### Question 2

Using the graph from the previous question, add nodes attributes named `'type'` where movies have the value `'movie'` and employees have the value `'employee'` and return that graph.

*This function should return a networkx graph with node attributes `{'type': 'movie'}` or `{'type': 'employee'}`*

In [15]:
def answer_two():
    
    # Your Code Here
    G = answer_one()
    n = set(G.nodes())
    e = set(['Andy', 'Claude', 'Frida', 'Georgia', 'Joan', 'Lee', 'Pablo', 'Vincent'])
    m = n - e
    G.add_nodes_from(e, type='employee')
    G.add_nodes_from(m, type='movie')
    #for (e, m) in G.edges():
    #    if G.node[e].get('type', None) == None:
    #        G.add_node(e, type='employee')
    #    if G.node[m].get('type', None) == None:
    #        G.add_node(m, type='movie')
    return G # Your Answer Here

#G = answer_two()
#print(G.nodes(data=True))
#plot_graph(G)

### Question 3

Find a weighted projection of the graph from `answer_two` which tells us how many movies different pairs of employees have in common.

*This function should return a weighted projected graph.*

In [16]:
def answer_three():
        
    # Your Code Here
    G = answer_two()
    #movies = set([n for n in G.nodes() if G.node[n]['type']=='movie'])
    #return bipartite.weighted_projected_graph(G, movies) # Your Answer Here
    employees = set([n for n in G.nodes() if G.node[n]['type']=='employee'])
    return bipartite.weighted_projected_graph(G, employees) # Your Answer Here
#G = answer_three()
#plot_graph(G)

### Question 4

Suppose you'd like to find out if people that have a high relationship score also like the same types of movies.

Find the Pearson correlation ( using `DataFrame.corr()` ) between employee relationship scores and the number of movies they have in common. If two employees have no movies in common it should be treated as a 0, not a missing value, and should be included in the correlation calculation.

*This function should return a float.*

In [34]:
def answer_four():
        
    # Your Code Here
    df = pd.read_csv("Employee_Relationships.txt", sep="\t", header=None)
    df.columns = ['e1', 'e2', 'score']
    df1 = df.copy()
    df1['e1'], df1['e2'] = df['e2'], df['e1']
    df = df.append(df1)
    #print(df)
    G = answer_three()
    df1 = pd.DataFrame()
    for (e1, e2) in G.edges():
        df1 = df1.append({'e1': e1, 'e2': e2, 'weight': G.edge[e1][e2]['weight']}, ignore_index=True)
        df1 = df1.append({'e1': e2, 'e2': e1, 'weight': G.edge[e1][e2]['weight']}, ignore_index=True)
    #print(df1) 
    df = df.merge(df1, left_on=['e1', 'e2'], right_on=['e1', 'e2'], how='outer')
    df.fillna(0, inplace=True)
    #print(df)
    return df['score'].corr(df['weight'], method='pearson') # Your Answer Here
#answer_four()